# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:
# Hücre 1: Drive’ı monte etme (eğer verileriniz Google Drive’da)

# Temel kütüphaneler
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    BatchNormalization,
    Dense,
    Dropout
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau,
    ModelCheckpoint,
    LearningRateScheduler
)

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils import class_weight

# GPU’nun varlığını kontrol edelim
print("TensorFlow sürümü:", tf.__version__)
print("Kullanılabilir GPU aygıtları:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Hücre 2: Sabitler ve veri dizinleri

# Görüntü boyutu ve batch size
IMG_SIZE       = 224        # DenseNet121 için standart boyut
BATCH_SIZE     = 32         # GPU belleğinize göre 16 veya 32 deneyebilirsiniz

# Epoch sayıları (Stage 1 + Stage 2 + Stage 3 = 120)
EPOCHS_STAGE1  = 10         # Sadece classifier head eğitimi
EPOCHS_STAGE2  = 90         # İnce ayar (Fine‐tuning) – ilk kısım
EPOCHS_STAGE3  = 20         # İnce ayar – son kısım
TOTAL_EPOCHS   = EPOCHS_STAGE1 + EPOCHS_STAGE2 + EPOCHS_STAGE3  # 120

# Öğrenme oranları
INITIAL_LR     = 1e-4       # Stage 1 için başlangıç LR
FINE_TUNE_LR   = 5e-5       # Stage 2 için ince ayar LR
FINAL_LR       = 1e-5       # Stage 3 için yeniden yükseltilmiş LR

# Veri dizinleri (kendi klasör yapınıza göre düzenleyin)
BASE_DIR       = 'data/split'
train_dir      = os.path.join(BASE_DIR, 'train')   # içinde SINIF1, SINIF2, SINIF3
test_dir       = os.path.join(BASE_DIR, 'test')    # içinde SINIF1, SINIF2, SINIF3

print("Train dizini:", train_dir)
print("Test dizini: ", test_dir)


In [ ]:
# Hücre 3: Eğitim/Doğrulama için ImageDataGenerator; test için ayrı generator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15,       # Eğitim verisinin %15’i doğrulama için ayrılacak
    rotation_range=20,           # 0–20 derece arası rasgele döndürme
    width_shift_range=0.15,      # Yatayda %15 kaydırma
    height_shift_range=0.15,     # Dikeyde %15 kaydırma
    shear_range=0.1,             # Shear dönüşümü
    zoom_range=[0.8, 1.2],       # %80–%120 arasında zoom
    brightness_range=[0.8, 1.2], # Parlaklık aralığı
    horizontal_flip=True,        # Yatayda çevirme
    fill_mode='nearest'          # Eksik pikselleri en yakın komşudan doldur
)

# 1) Train generator (%85’lik eğitim kısmı)
train_generator = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',           # validation_split=0.15 => %85’lik kısım
    shuffle=True
)

# 2) Validation generator (%15’lik doğrulama kısmı)
val_generator = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',         # validation_split=0.15 => %15’lik kısım
    shuffle=True
)

# 3) Test generator (ayrı test klasörü, augmentasyon yok; sadece rescale)
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    directory=test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
# Hücre 4:
from sklearn.utils import class_weight

y_train = train_generator.classes
cw = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: w for i, w in enumerate(cw)}
print("Hesaplanan class_weights:", class_weights)


In [ ]:
# Hücre 5: DenseNet121 base model’i yükleyelim; tüm katmanları donduralım
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 1) Base model’in tüm katmanlarını “trainable=False” yapıyoruz:
for layer in base_model.layers:
    layer.trainable = False

# 2) Üst sınıflayıcı (classifier head) katmanlarını ekleyelim:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)

# - 1. Dense katman + L2 + ReLU + BatchNorm + Dropout
x = Dense(256, kernel_regularizer=l2(1e-3), activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

# - 2. Dense katman + L2 + ReLU + BatchNorm + Dropout
x = Dense(128, kernel_regularizer=l2(1e-4), activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

# - Son katman (softmax, dinamik sınıf sayısı)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

# 3) Modeli oluştur
model = Model(inputs=base_model.input, outputs=predictions)

# 4) Model özetini gör
model.summary()


In [ ]:
# Hücre 6: Stage 1 için callback’leri ve compile işlemi

# 1) ReduceLROnPlateau: validation loss platoya girerse LR’ı yarıya düşürelim
reduce_lr_1 = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1,
    min_lr=1e-7
)

# 2) ModelCheckpoint: validation accuracy yükseldiğinde ağırlıkları kaydedelim
checkpoint_1 = ModelCheckpoint(
    filepath='best_densenet121_stage1.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# 3) LearningRateScheduler (opsiyonel): epoch 5’te LR’ı yarıya düşürelim
def lr_schedule_1(epoch, lr):
    if epoch == 5:
        return lr * 0.5
    return lr

lr_scheduler_1 = LearningRateScheduler(lr_schedule_1, verbose=1)

callbacks_stage1 = [reduce_lr_1, checkpoint_1, lr_scheduler_1]

# 4) Modeli compile edelim (Stage 1: sadece başlık eğitimi)
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Hücre 7: Modelin sadece başlık katmanlarını eğitiyoruz

history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE1,           # 10 epoch
    callbacks=callbacks_stage1,
    class_weight=class_weights      # Class imbalance için
)


In [ ]:
# Hücre 8: Stage 2’de base_model’in son katmanlarını açıp fine-tuning’e geçiyoruz

# 1) Son 80 katmanı açalım (örnek olarak)
for layer in base_model.layers[:80]:
    layer.trainable = False
for layer in base_model.layers[80:]:
    layer.trainable = True

# 2) Modeli yeniden derleyelim (daha küçük bir LR kullanalım)
model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Hücre 9: Stage 2 için callback’leri tanımlayalım ve eğitime başlayalım

# 1) ReduceLROnPlateau (Stage 2)
reduce_lr_2 = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1,
    min_lr=1e-8
)

# 2) ModelCheckpoint (Stage 2)
checkpoint_2 = ModelCheckpoint(
    filepath='best_densenet121_stage2.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# 3) LearningRateScheduler: 30. ve 60. epoch’larda LR’ı tekrar FINE_TUNE_LR’e yükseltelim
def lr_schedule_2(epoch, lr):
    # Eğer epoch index’i  EPOCHS_STAGE1 + 30 (yani 10 + 30 = 40) veya 10 + 60 = 70 ise yeniden 5e-5
    if epoch == EPOCHS_STAGE1 + 30 or epoch == EPOCHS_STAGE1 + 60:
        return FINE_TUNE_LR
    return lr

lr_scheduler_2 = LearningRateScheduler(lr_schedule_2, verbose=1)

callbacks_stage2 = [reduce_lr_2, checkpoint_2, lr_scheduler_2]

# 4) Eğitime başla (Stage 2: 90 epoch; initial_epoch = 10’dan devam)
history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE1 + EPOCHS_STAGE2,    # 10 + 90 = 100 → 0..99 arası
    initial_epoch=history_stage1.epoch[-1] + 1,  # 10’dan (indeks 10) devam
    callbacks=callbacks_stage2,
    class_weight=class_weights
)


In [ ]:
# Hücre 10: Stage 3 – 100..119 arasındaki 20 epoch boyunca son ek ayarlar

# 1) Bu kez baz modeli biraz daha geniş açalım: son 60 katmanı eğitime aç
for layer in base_model.layers[:60]:
    layer.trainable = False
for layer in base_model.layers[60:]:
    layer.trainable = True

# 2) Öğrenme oranını biraz daha yükseltelim (1e-5)
model.compile(
    optimizer=Adam(learning_rate=FINAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3) Yeni ReduceLROnPlateau ve ModelCheckpoint callback’leri (Stage 3 için)
reduce_lr_3 = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,     # Daha sabırsız: 3 epoch içinde iyileşme yoksa LR düş
    verbose=1,
    min_lr=1e-8
)

checkpoint_3 = ModelCheckpoint(
    filepath='best_densenet121_stage3.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

callbacks_stage3 = [reduce_lr_3, checkpoint_3]

# 4) Stage 3’ü çalıştıralım (100..119 arası, 20 epoch)
history_stage3 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=TOTAL_EPOCHS,           # 120 → dizin 0..119 tamamlanacak
    initial_epoch=history_stage2.epoch[-1] + 1,  # 100. epoch’dan (indeks 100) başla
    callbacks=callbacks_stage3,
    class_weight=class_weights
)


In [ ]:
# Hücre 11: Stage 1 + Stage 2 + Stage 3 toplamını birleştirip grafik çizelim

# 1) Her aşamanın history’sini alın
h1 = history_stage1.history
h2 = history_stage2.history
h3 = history_stage3.history

# 2) loss, val_loss, accuracy, val_accuracy listelerini ardışık birleştir
merged_history = {
    'loss':         h1['loss']         + h2['loss']         + h3['loss'],
    'val_loss':     h1['val_loss']     + h2['val_loss']     + h3['val_loss'],
    'accuracy':     h1['accuracy']     + h2['accuracy']     + h3['accuracy'],
    'val_accuracy': h1['val_accuracy'] + h2['val_accuracy'] + h3['val_accuracy']
}

# 3) Kayıp grafiği
plt.figure(figsize=(8, 6))
plt.plot(merged_history['loss'],      label='Eğitim Kaybı',     linewidth=2)
plt.plot(merged_history['val_loss'],  label='Doğrulama Kaybı',  linewidth=2)
plt.title('Model Kayıp Eğrisi (0–119 Epoch)')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()
plt.grid(True)
plt.show()

# 4) Doğruluk grafiği
plt.figure(figsize=(8, 6))
plt.plot(merged_history['accuracy'],     label='Eğitim Doğruluğu',     linewidth=2)
plt.plot(merged_history['val_accuracy'], label='Doğrulama Doğruluğu', linewidth=2)
plt.title('Model Doğruluk Eğrisi (0–119 Epoch)')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Hücre 12: En iyi Stage 3 ağırlıklarını yükleyip test seti üzerinde değerlendirelim

# 1) En iyi Stage 3 modelini yükle
model.load_weights('best_densenet121_stage3.h5')

# 2) Test setindeki olasılık tahminlerini al
y_pred_probs = model.predict(test_generator, verbose=1)

# 3) Her örnek için en yüksek olasılıklı sınıfı belirle
y_pred = np.argmax(y_pred_probs, axis=1)

# 4) Gerçek etiketleri al
y_true = test_generator.classes

# 5) Sınıf adlarını almak için ters harita
class_indices = train_generator.class_indices  # {'SINIF1':0, 'SINIF2':1, 'SINIF3':2}
idx_to_class = {v: k for k, v in class_indices.items()}

# 6) Classification report
print("=== Classification Report ===\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[idx_to_class[i] for i in range(len(idx_to_class))]
))

# 7) Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=[idx_to_class[i] for i in range(len(idx_to_class))],
    yticklabels=[idx_to_class[i] for i in range(len(idx_to_class))]
)
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Test Seti Karışıklık Matrisi')
plt.show()


In [ ]:
# Hücre 13: 3 sınıflı ROC eğrisi çizimi (micro-average + her sınıf)

# 1) One-hot gerçek etiketler
n_classes = train_generator.num_classes  # 3
y_true_onehot = label_binarize(y_true, classes=list(range(n_classes)))

# 2) Her sınıf için FPR/TPR/AUC hesapla
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_onehot[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 3) Mikro-ortalama ROC
fpr["micro"], tpr["micro"], _ = roc_curve(
    y_true_onehot.ravel(),
    y_pred_probs.ravel()
)
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# 4) Grafik çiz
plt.figure(figsize=(8, 6))

# Mikro-ortalama eğrisi
plt.plot(
    fpr["micro"],
    tpr["micro"],
    label=f'micro-average ROC (AUC = {roc_auc["micro"]:.2f})',
    color='magenta',
    linestyle=':',
    linewidth=3
)

# Her sınıf için ROC eğrisi
colors = ['aqua', 'darkorange', 'cornflowerblue']
for i, color in zip(range(n_classes), colors):
    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f'ROC of {idx_to_class[i]} (AUC = {roc_auc[i]:.2f})'
    )

# 45° baz çizgisi
plt.plot([0, 1], [0, 1], 'k--', lw=1)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Çok Sınıflı ROC Eğrileri')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
# Hücre 14 (Güncellenmiş): Modeli Kaydetme
# -------------------------------

import shutil

# 1) HDF5 (.h5) olarak kaydetme
#    Uyarıya rağmen bu satır çalışacak ve final_densenet121.h5 dosyasını oluşturacaktır.
save_path_h5 = 'artifacts/densenet121_son.h5'
model.save(save_path_h5)
print("Model (HDF5) şu adrese kaydedildi:\n", save_path_h5)







In [ ]:
#modeli çalıştırma
h5_path = 'artifacts/densenet121_son.h5'
from tensorflow.keras.models import load_model

# 1) Kaydedilmiş HDF5 modelini yükleyelim
h5_path = 'artifacts/densenet121_son.h5'
model = load_model(h5_path)

print("📋 Model başarıyla yüklendi. Aşağıda katman yapısı (summary) var:")
model.summary()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# ----- 0. Sınıf isimlerini (target_names) oluştur -----
# test_generator.class_indices: {'tip1': 0, 'tip2': 1, 'tip3': 2, ...} gibi bir sözlük döner.
# Biz bu sözlüğü indeks sırasına göre sıralayıp bir listeye koyacağız:
#
# Örneğin:
#   test_generator.class_indices = {'tip2': 1, 'tip1': 0, 'tip3': 2}
#   sorted(test_generator.class_indices.items(), key=lambda x: x[1])
#   → [('tip1', 0), ('tip2', 1), ('tip3', 2)]
#
# Böylece indeks sırasına göre isimleri elde etmiş oluyoruz.

target_names = [class_name for class_name, _ in
                sorted(test_generator.class_indices.items(), key=lambda x: x[1])]

# ----- 1. Rastgele bir test resmi seç -----
file_paths = test_generator.filepaths
random_index = random.randint(0, len(file_paths) - 1)
img_path = file_paths[random_index]

# ----- 2. Görüntüyü yükle ve normalize et -----
# IMG_SIZE: Daha önce tanımlanmış olmalı (örneğin 224, 299 vb.).
# Eğer IMG_SIZE kullanılmıyorsa, IMG_HEIGHT ve IMG_WIDTH kullanacak şekilde ayarla.
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# ----- 3. Model ile tahmin yap ve sınıf indekslerini al -----
pred_probs = model.predict(img_array)[0]        # Örneğin: [0.10, 0.70, 0.20]
predicted_idx = np.argmax(pred_probs)
predicted_class = target_names[predicted_idx]

true_idx = test_generator.classes[random_index]
true_class = target_names[true_idx]

# ----- 4. Sonuçları ekrana bastır -----
print(f"Gerçek:   {true_class}")
print(f"Tahmin:   {predicted_class}")
print()  # boş bir satır
for i, name in enumerate(target_names):
    print(f"{name}: {pred_probs[i]:.2f}")

# ----- 5. Alt kısımda yalnızca resmi göster -----
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.show()